<a href="https://colab.research.google.com/github/ramneekh/masteringAI/blob/GenAI/Assignment_12_Spam_vs_Ham_Email_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Classify emails as spam or ham using an LSTM/RNN model based on the dataset in "Ramneek Handa - SPAM text message 20170820 - Data - Ramneek Handa - SPAM text message 20170820 - Data.csv".

## Load and explore the data

### Subtask:
Load the dataset into a pandas DataFrame and perform some initial exploration to understand the data structure, check for missing values, and examine the distribution of spam and ham messages.


**Reasoning**:
Load the dataset and perform initial data exploration steps including displaying head, info, checking for missing values, and unique values.



In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/Ramneek Handa - SPAM text message 20170820 - Data - Ramneek Handa - SPAM text message 20170820 - Data.csv', encoding='latin-1')

# Display the first 5 rows
print("First 5 rows of the DataFrame:")
display(df.head())

# Print column names and data types
print("\nColumn names and data types:")
display(df.info())

# Check for missing values
print("\nMissing values per column:")
display(df.isnull().sum())

# Get number of unique values
print("\nNumber of unique values per column:")
display(df.nunique())

# Display the distribution of the target variable
print("\nDistribution of spam and ham messages:")
display(df['v1'].value_counts())

First 5 rows of the DataFrame:


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



Column names and data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


None


Missing values per column:


,0
Category,0
Message,0



Number of unique values per column:


,0
Category,2
Message,5157



Distribution of spam and ham messages:


KeyError: 'v1'

**Reasoning**:
The previous code failed because the column name for the target variable is 'Category' instead of 'v1'. I need to correct the column name to display the distribution of spam and ham messages.



In [3]:
# Display the distribution of the target variable
print("\nDistribution of spam and ham messages:")
display(df['Category'].value_counts())


Distribution of spam and ham messages:


,count
Category,
ham,4825
spam,747


## Preprocess the text data

### Subtask:
Clean the text data by removing special characters, punctuation, and converting text to lowercase. Tokenize the text and convert it into sequences of numbers that can be used as input for the LSTM/RNN model.


**Reasoning**:
Define a function to clean the text data, apply it to the 'Message' column, initialize a Tokenizer, fit it, convert text to sequences, pad the sequences, and convert the 'Category' column to numerical labels.



In [5]:
import re
import string
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Define a function to clean the text data
def clean_text(text):
    text = text.lower() # Convert to lowercase
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text) # Remove punctuation
    text = re.sub(r'\d+', '', text) # Remove numbers
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra whitespace
    return text

# 2. Apply the cleaning function to the 'Message' column
df['Cleaned_Message'] = df['Message'].apply(clean_text)

# 3. Initialize a Tokenizer object
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")

# 4. Fit the tokenizer on the cleaned text data
tokenizer.fit_on_texts(df['Cleaned_Message'])

# 5. Convert the cleaned text data into sequences of integers
sequences = tokenizer.texts_to_sequences(df['Cleaned_Message'])

# 6. Pad the sequences
max_length = 100 # Determine an appropriate maximum sequence length
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

# 7. Store the padded sequences in a new variable (already done in previous step)
# padded_sequences is the new variable

# 8. Convert the 'Category' column to numerical labels
df['Category_Numerical'] = df['Category'].map({'ham': 0, 'spam': 1})

# Display the first few rows of the DataFrame with new columns and the padded sequences shape
print("DataFrame with cleaned text and numerical category:")
display(df.head())
print("\nShape of padded sequences:")
print(padded_sequences.shape)

DataFrame with cleaned text and numerical category:


,Category,Message,Cleaned_Message,Category_Numerical
0,ham,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,0
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final ...,1
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,0



Shape of padded sequences:
(5572, 100)


## Split the data

### Subtask:
Split the data into training and testing sets to evaluate the model's performance on unseen data.


**Reasoning**:
Split the data into training and testing sets using the padded sequences as features and the numerical category as the target.



In [7]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = padded_sequences
y = df['Category_Numerical']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Print the shapes of the resulting sets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (4457, 100)
Shape of X_test: (1115, 100)
Shape of y_train: (4457,)
Shape of y_test: (1115,)


## Build the lstm/rnn model

### Subtask:
Define the architecture of the LSTM/RNN model, including the embedding layer, LSTM/RNN layers, and output layer.


**Reasoning**:
Define the architecture of the LSTM model as specified in the instructions, including the embedding layer, LSTM layer, and dense output layer, then print the model summary.



In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Get the vocabulary size from the tokenizer
vocab_size = len(tokenizer.word_index) + 1 # Add 1 for the OOV token

# Define the model architecture
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    LSTM(units=64),
    Dense(1, activation='sigmoid')
])

# Print the model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Compile and train the model

### Subtask:
Compile the model with an appropriate loss function and optimizer, and train it on the training data.


**Reasoning**:
Compile and train the defined model using the specified loss function, optimizer, metrics, epochs, and batch size.



In [9]:
# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

# Display the training history keys to verify
print(history.history.keys())

Epoch 1/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.8490 - loss: 0.4129 - val_accuracy: 0.8430 - val_loss: 0.4360
Epoch 2/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.8770 - loss: 0.3743 - val_accuracy: 0.8430 - val_loss: 0.4346
Epoch 3/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 48ms/step - accuracy: 0.8734 - loss: 0.3823 - val_accuracy: 0.8430 - val_loss: 0.4367
Epoch 4/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - accuracy: 0.8736 - loss: 0.3803 - val_accuracy: 0.8430 - val_loss: 0.4353
Epoch 5/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.8804 - loss: 0.3671 - val_accuracy: 0.8430 - val_loss: 0.4365
Epoch 6/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.8738 - loss: 0.3797 - val_accuracy: 0.8430 - val_loss: 0.4398
Epoch 7/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.8758 - loss: 0.3753 - val_accuracy: 0.8430 - val_loss: 0.4501
Epoch 8/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 11s 50ms/step - accuracy: 0.8733 - loss: 0.3822 - val_ac

## Evaluate the model

### Subtask:
Evaluate the trained model on the testing data to assess its performance using metrics such as accuracy, precision, recall, and F1-score.


**Reasoning**:
Evaluate the trained model on the testing data using accuracy, precision, recall, and F1-score, and print the results.



In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Make predictions on the testing data
y_pred_prob = model.predict(X_test)

# Convert predicted probabilities to binary class labels (0 or 1) using a threshold of 0.5
y_pred = (y_pred_prob >= 0.5).astype(int)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Print the evaluation metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step
Accuracy: 0.8664
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Make predictions

### Subtask:
Use the trained model to make predictions on new, unseen text messages.


**Reasoning**:
Create a list of new messages, preprocess them using the fitted tokenizer and max_length, predict using the trained model, convert predictions to binary labels, and print the results.



In [11]:
# 1. Create a list of new text messages
new_messages = [
    "Congratulations! You've won a free iPhone! Click here to claim.", # Expected Spam
    "Hey, just wanted to confirm our meeting time tomorrow.", # Expected Ham
    "URGENT! Your bank account has been compromised. Click the link below to secure it.", # Expected Spam
    "Can we reschedule our call for later this week?", # Expected Ham
    "Free access to exclusive content! Limited time offer!", # Expected Spam
    "Looking forward to seeing you at the party tonight.", # Expected Ham
]

# 2. Preprocess the new messages
cleaned_new_messages = [clean_text(msg) for msg in new_messages]
new_sequences = tokenizer.texts_to_sequences(cleaned_new_messages)
padded_new_sequences = pad_sequences(new_sequences, maxlen=max_length, padding='post', truncating='post')

# 3. Use the trained model's .predict() method
new_predictions_prob = model.predict(padded_new_sequences)

# 4. Convert probability predictions into binary class labels
new_predictions = (new_predictions_prob >= 0.5).astype(int)

# 5. Print the original new messages along with their predicted labels
print("New Message Predictions:")
for i, message in enumerate(new_messages):
    predicted_label = "spam" if new_predictions[i][0] == 1 else "ham"
    print(f"Message: '{message}'")
    print(f"Predicted Label: {predicted_label}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
New Message Predictions:
Message: 'Congratulations! You've won a free iPhone! Click here to claim.'
Predicted Label: ham

Message: 'Hey, just wanted to confirm our meeting time tomorrow.'
Predicted Label: ham

Message: 'URGENT! Your bank account has been compromised. Click the link below to secure it.'
Predicted Label: ham

Message: 'Can we reschedule our call for later this week?'
Predicted Label: ham

Message: 'Free access to exclusive content! Limited time offer!'
Predicted Label: ham

Message: 'Looking forward to seeing you at the party tonight.'
Predicted Label: ham



## Summary:

### Data Analysis Key Findings

*   The dataset contains emails categorized as 'ham' and 'spam', with a significant imbalance (more ham than spam messages). Specifically, there are 4825 ham messages and 747 spam messages.
*   The text data was cleaned by converting to lowercase, removing punctuation, numbers, and extra whitespace.
*   The cleaned text data was tokenized, creating a vocabulary of 10000 words (including an out-of-vocabulary token).
*   Text sequences were padded to a uniform length of 100.
*   The category labels were converted to numerical values (0 for 'ham', 1 for 'spam').
*   The dataset was split into training (80%) and testing (20%) sets, maintaining the class distribution using stratification. The training set has 4457 samples, and the testing set has 1115 samples.
*   An LSTM model was built with an embedding layer (vocabulary size + 1, output dimension 128), an LSTM layer (64 units), and a dense output layer with sigmoid activation.
*   The model was compiled using `binary_crossentropy` loss, `adam` optimizer, and 'accuracy' metric.
*   The model was trained for 10 epochs with a batch size of 32. Training accuracy generally increased, while validation accuracy remained relatively stable.
*   Model evaluation on the test set yielded an accuracy of 0.8664. However, the precision, recall, and F1-score for the 'spam' class were all 0.0000, indicating the model did not correctly classify any spam messages in the test set.
*   When tested on new messages, the model predicted all provided examples, including those expected to be spam, as 'ham'.

### Insights or Next Steps

*   The significant class imbalance between ham and spam messages likely contributed to the model's inability to detect the minority class (spam). Addressing this imbalance through techniques like oversampling the minority class, undersampling the majority class, or using weighted loss functions could improve spam detection.
*   Further investigation into the model architecture, hyperparameters (e.g., number of LSTM units, learning rate, number of epochs), and potentially exploring other RNN architectures or attention mechanisms might improve the model's performance on the spam class.
